# 0. Steerable Pyramid Feature Extraction

**Purpose:**  
This notebook computes **steerable pyramid feature maps** for all images.

These features serve as the input representation for the encoding model.

---

## Output Usage (Next Step)

The outputs generated in this notebook are required for:

- `1_nsd_prf_sampling_from_pyramid.ipynb`  
  → Performs pRF sampling to convert feature maps into voxel-specific features

This notebook must be completed before running the pRF sampling step.

---
## What This Notebook Does

For each image:

- Applies a **steerable pyramid decomposition**
- Extracts:
  - Multiple spatial frequency levels
  - Multiple orientations per level
- Produces feature maps representing image structure

The outputs are saved as `.npz` files, one per image.

Each file contains:

- `sumOri`  
  → Feature maps summed across orientations

- `modelOri`  
  → Feature maps separated by orientation and level

  ---

## Using This Notebook with New Data

To apply this pipeline to a new dataset, you must provide:

1. A set of images in hDF5 format
2. A consistent naming or indexing scheme
3. A folder to save `.npz` outputs

Each image will be converted into a steerable pyramid representation,
which will later be used for pRF sampling.

## Implementation Details

The steerable pyramid decomposition is implemented using the `pyrtools` library:

https://pyrtools.readthedocs.io/en/latest/

This library provides an efficient implementation of multi-scale, multi-orientation image decompositions used to extract visual features.

The outputs of this step (`sumOri`, `modelOri`) follow the structure produced by this library.

## Code Execution

In [ ]:


import os
import json
import time
import numpy as np
import cv2
import h5py
import pyrtools as pt
import matplotlib.pyplot as plt
import scipy.io as sio


# ============================================================
# NSD design helpers
# ============================================================

def _load_nsd_design(nsd_design_mat_path):
    mat = sio.loadmat(nsd_design_mat_path, squeeze_me=True)
    nsdDesign = mat.get("nsdDesign", mat)

    # Handles both dict-like and MATLAB-struct-like loads
    subjectim = nsdDesign["subjectim"]
    masterordering = nsdDesign["masterordering"].astype(int)

    return subjectim, masterordering


def _subject_image_set(subjectim, masterordering, subject_id):
    """
    Return unique image indices, converted from MATLAB 1-based image IDs
    to Python 0-based image IDs. 
    Notes:
    - NSD indices are 1-based in the MAT.
    - Zeros can appear as padding -> remove.
    """
    # subjectim: (n_subs, n_trials) with 1-based image IDs (or 0 padding)
    # masterordering: length n_trials, 1-based positions
    subj = subjectim[subject_id, masterordering - 1] # still 1-based image IDs, may include 0
    subj = np.asarray(subj).astype(np.int64)

    subj = subj[subj > 0] # remove zeros/padding
    subj = np.unique(subj) 

    return sorted(map(int, subj - 1))


# ============================================================
# Configurable stimulus processor
# ============================================================

class StimulusPyramidProcessor:
    """
    General stimulus processor.

    This version control the behavior through configuration options:

    - resize method
    - interpolation size
    - background yes/no
    - fixation yes/no
    - fixation size/color/shape/alpha
    - grayscale method
    - optional 0-1 scaling
    - pyramid input size
    - number of orientations
    - number of levels

    Important:
    This code does NOT z-score the image before the steerable pyramid.
    """

    def __init__(
        self,
        stim_filename,
        output_folder,

        # image resizing
        interp_img_size=256,
        resize_method="resize",          # "resize" or "remap"

        # background
        use_background=False,
        background_size=1024,
        background_color=127,

        # fixation
        use_fixation=True,
        fixation_shape="square",         # "square" or "circle"
        fixation_size=None,              # if None, auto-scale from 17/714
        fixation_ref_size=714,
        fixation_ref_patch=17,
        fixation_color=(255, 0, 0),
        fixation_alpha=0.5,

        # grayscale
        grayscale_method="bt709",        # "bt709" or "mean_rgb"
        normalize_to_unit=False,         # divide RGB by 255 before grayscale

        # pyramid
        pyramid_input_size=256,
        num_orientations=4,
        num_levels=4,                    # integer or "auto"
        bandwidth=1,
        allow_high_levels=False,

        # run control
        max_images=None,
    ):
        self.stim_filename = stim_filename
        self.output_folder = output_folder
        os.makedirs(self.output_folder, exist_ok=True)

        self.interp_img_size = int(interp_img_size)
        self.resize_method = resize_method

        self.use_background = use_background
        self.background_size = int(background_size)
        self.background_color = background_color

        self.use_fixation = use_fixation
        self.fixation_shape = fixation_shape
        self.fixation_size = fixation_size
        self.fixation_ref_size = fixation_ref_size
        self.fixation_ref_patch = fixation_ref_patch
        self.fixation_color = np.array(fixation_color, dtype=np.float32)
        self.fixation_alpha = float(fixation_alpha)

        self.grayscale_method = grayscale_method
        self.normalize_to_unit = normalize_to_unit

        self.pyramid_input_size = int(pyramid_input_size)
        self.dims = [self.pyramid_input_size, self.pyramid_input_size]

        self.num_orientations = int(num_orientations)
        self.num_levels_setting = num_levels
        self.num_levels = None
        self.bandwidth = bandwidth
        self.allow_high_levels = allow_high_levels

        self.allowed_img_indices = None

        with h5py.File(self.stim_filename, "r") as f:
            dataset_shape = f["imgBrick"].shape
            self.total_images = dataset_shape[0]
            self.img_size_x, self.img_size_y, _ = dataset_shape[1:]

        self.max_images = self.total_images if max_images is None else min(max_images, self.total_images)

        if self.resize_method == "remap":
            self.xq, self.yq = np.meshgrid(
                np.linspace(0, self.img_size_x - 1, self.interp_img_size),
                np.linspace(0, self.img_size_y - 1, self.interp_img_size),
            )
            self.xq = self.xq.astype(np.float32)
            self.yq = self.yq.astype(np.float32)

        print(
            f"Dataset contains {self.total_images} images. "
            f"Configured max_images={self.max_images}. "
            f"Output folder: {self.output_folder}"
        )

    # ------------------------------------------------------------
    # Design/image selection
    # ------------------------------------------------------------

    def set_allowed_images_from_design(
        self,
        nsd_design_mat_path,
        subject_id=None,
        union_across_subjects=False,
    ):
        """
        Correct behavior options:
        - subject_id is not None: restrict to ~10,000 images for that subject (recommended).
        - union_across_subjects=True: union across all 8 subjects (can be large; not recommended).
        (Depending on NSD design, union can approach the full 73k pool.)
        """
        
        subjectim, masterordering = _load_nsd_design(nsd_design_mat_path)
        n_subs = subjectim.shape[0]

        if subject_id is not None:
            imgs = _subject_image_set(subjectim, masterordering, subject_id)
            self.allowed_img_indices = imgs
            print(f"Using {len(imgs)} images for subject {subject_id + 1}")
            return

        if union_across_subjects:
            all_imgs = set()
            for sid in range(n_subs):
                all_imgs.update(_subject_image_set(subjectim, masterordering, sid))
            self.allowed_img_indices = sorted(all_imgs)
            print(f"Using {len(self.allowed_img_indices)} images: union across subjects")
            return

        raise ValueError("Provide subject_id or set union_across_subjects=True.")

    # ------------------------------------------------------------
    # Basic image operations
    # ------------------------------------------------------------

    def load_image(self, img_num, imgbrick=None):
        if img_num >= self.total_images:
            raise IndexError(
                f"Image index {img_num} out of range 0-{self.total_images - 1}"
            )

        if imgbrick is not None:
            return np.array(imgbrick[img_num], dtype=np.float32)

        with h5py.File(self.stim_filename, "r") as f:
            return np.array(f["imgBrick"][img_num], dtype=np.float32)

    def resize_image(self, img):
        """
        Resize original NSD image to interp_img_size.

        resize_method="resize":
            direct cv2.resize, useful for 425 -> 256.

        resize_method="remap":
            MATLAB-like interpolation grid, useful for 425 -> 714.
        """
        if self.resize_method == "resize":
            return cv2.resize(
                img,
                (self.interp_img_size, self.interp_img_size),
                interpolation=cv2.INTER_LINEAR,
            ).astype(np.float32)

        elif self.resize_method == "remap":
            interp_img = np.zeros(
                (self.interp_img_size, self.interp_img_size, 3),
                dtype=np.float32,
            )

            for c in range(3):
                interp_img[:, :, c] = cv2.remap(
                    img[:, :, c].astype(np.float32),
                    self.xq,
                    self.yq,
                    interpolation=cv2.INTER_LINEAR,
                )

            return interp_img

        else:
            raise ValueError("resize_method must be 'resize' or 'remap'.")

    def _get_fixation_size(self):
        if self.fixation_size is not None:
            size = int(self.fixation_size)
        else:
            size = int(round(
                self.interp_img_size * (self.fixation_ref_patch / self.fixation_ref_size)
            ))

        if size < 1:
            size = 1

        # Force odd size for symmetric square fixation
        if size % 2 == 0:
            size += 1

        return size

    def add_fixation_point(self, img):
        """
        Add semi-transparent fixation marker to RGB image.

        Supports:
        - square fixation
        - circular fixation
        """
        img = img.astype(np.float32)

        size = self._get_fixation_size()
        center = self.interp_img_size // 2

        overlay = img.copy()

        if self.fixation_shape == "square":
            half = size // 2
            r0, r1 = center - half, center + half + 1
            c0, c1 = center - half, center + half + 1

            overlay[r0:r1, c0:c1, :] = self.fixation_color[None, None, :]

        elif self.fixation_shape == "circle":
            radius = size // 2
            cv2.circle(
                overlay,
                (center, center),
                radius,
                tuple(float(x) for x in self.fixation_color),
                thickness=-1,
            )

        else:
            raise ValueError("fixation_shape must be 'square' or 'circle'.")

        img = (
            self.fixation_alpha * overlay
            + (1.0 - self.fixation_alpha) * img
        )

        return img.astype(np.float32)

    def add_background(self, img):
        """
        Place image in the center of a larger gray RGB background.
        """
        if self.background_size < self.interp_img_size:
            raise ValueError("background_size must be >= interp_img_size.")

        big_img = np.full(
            (self.background_size, self.background_size, 3),
            self.background_color,
            dtype=np.float32,
        )

        start = (self.background_size - self.interp_img_size) // 2
        end = start + self.interp_img_size

        big_img[start:end, start:end, :] = img.astype(np.float32)

        return big_img

    def to_grayscale(self, img_rgb):
        """
        Convert RGB image to grayscale.

        grayscale_method="mean_rgb":
            average across RGB channels.

        grayscale_method="bt709":
            luminance-preserving BT.709 transform.

        normalize_to_unit=True:
            converts 0-255 to 0-1 before grayscale.
            This is not z-scoring.
        """
        x = img_rgb.astype(np.float32)

        if self.normalize_to_unit:
            x = x / 255.0

        if self.grayscale_method == "mean_rgb":
            gray = np.mean(x, axis=2)

        elif self.grayscale_method == "bt709":
            gray = (
                0.2126 * x[:, :, 0]
                + 0.7152 * x[:, :, 1]
                + 0.0722 * x[:, :, 2]
            )

        else:
            raise ValueError("grayscale_method must be 'mean_rgb' or 'bt709'.")

        return gray.astype(np.float32)

    def resize_for_pyramid(self, gray_img):
        """
        Resize grayscale image to final pyramid input size.
        """
        if gray_img.shape == tuple(self.dims):
            return gray_img.astype(np.float32)

        return cv2.resize(
            gray_img,
            (self.dims[1], self.dims[0]),
            interpolation=cv2.INTER_AREA,
        ).astype(np.float32)

    # ------------------------------------------------------------
    # Steerable pyramid
    # ------------------------------------------------------------

    def height(self, dims, bandwidth, allow_high_levels=False):
        max_ht = int(np.log2(min(dims)))
        num_levels = int(np.log2(min(dims)) - np.log2(bandwidth))

        if allow_high_levels:
            return min(num_levels, max_ht)

        return min(num_levels, 7)

    def get_num_levels(self, img_shape):
        if self.num_levels_setting == "auto":
            return self.height(
                img_shape,
                self.bandwidth,
                self.allow_high_levels,
            )

        return int(self.num_levels_setting)

    def apply_steerable_pyramid(self, gray_img):
        """
        Apply steerable pyramid.

        Important:
        - No image z-scoring is performed here.
        - pyrtools order = num_orientations - 1.
        """
        img_shape = gray_img.shape[:2]
        self.num_levels = self.get_num_levels(img_shape)

        if img_shape != tuple(self.dims):
            print(f"Warning: expected pyramid input {self.dims}, got {img_shape}")

        return pt.pyramids.SteerablePyramidFreq(
            gray_img.astype(np.float32),
            height=self.num_levels,
            order=self.num_orientations - 1,
        )

    # ------------------------------------------------------------
    # Main processing
    # ------------------------------------------------------------

    def process_image(self, img_num, imgbrick=None, norm_resp=False):
        start_time = time.time()

        save_path = os.path.join(self.output_folder, f"pyrImg{img_num}.npz")

        if os.path.exists(save_path):
            return

        # Load original image
        img = self.load_image(img_num, imgbrick=imgbrick)

        # Resize original image
        img = self.resize_image(img)

        # Optional fixation marker
        if self.use_fixation:
            img = self.add_fixation_point(img)

        # Optional background
        if self.use_background:
            img = self.add_background(img)

        # Convert to grayscale
        bigImg = self.to_grayscale(img)

        # Resize to pyramid input size
        bigImg = self.resize_for_pyramid(bigImg)

        # Apply pyramid
        pyr = self.apply_steerable_pyramid(bigImg)
        pyr_coeffs = pyr.pyr_coeffs

        # Build outputs
        sum_ori = [
            np.zeros_like(bigImg, dtype=np.float32)
            for _ in range(self.num_levels + 2)
        ]

        model_ori = [
            np.zeros(
                (self.num_orientations, *bigImg.shape),
                dtype=np.float32,
            )
            for _ in range(self.num_levels + 2)
        ]

        for ilev in range(self.num_levels):
            for iori in range(self.num_orientations):
                key = (ilev, iori)

                if key not in pyr_coeffs:
                    continue

                band = np.abs(pyr_coeffs[key]).astype(np.float32)

                band_resized = cv2.resize(
                    band,
                    (self.dims[1], self.dims[0]),
                    interpolation=cv2.INTER_LINEAR,
                )

                sum_ori[ilev] += band_resized
                model_ori[ilev][iori] = band_resized

        # Low-pass residual
        if "residual_lowpass" in pyr_coeffs:
            lowpass = np.abs(pyr_coeffs["residual_lowpass"]).astype(np.float32)
            lowpass = cv2.resize(
                lowpass,
                (self.dims[1], self.dims[0]),
                interpolation=cv2.INTER_LINEAR,
            )
            sum_ori[self.num_levels] = lowpass
            model_ori[self.num_levels] = lowpass

        # High-pass residual
        if "residual_highpass" in pyr_coeffs:
            highpass = np.abs(pyr_coeffs["residual_highpass"]).astype(np.float32)
            highpass = cv2.resize(
                highpass,
                (self.dims[1], self.dims[0]),
                interpolation=cv2.INTER_LINEAR,
            )
            sum_ori[self.num_levels + 1] = highpass
            model_ori[self.num_levels + 1] = highpass

        np.savez_compressed(
            save_path,
            bigImg=bigImg.astype(np.float32),
            sumOri=np.stack(sum_ori, axis=0).astype(np.float32),
            modelOri=np.array(model_ori, dtype=object),
            numLevels=np.array(self.num_levels, dtype=np.int32),
            interpImgSize=np.array(self.interp_img_size, dtype=np.int32),
            backgroundSize=np.array(
                self.background_size if self.use_background else 0,
                dtype=np.int32,
            ),
            useBackground=np.array(self.use_background, dtype=np.int8),
            useFixation=np.array(self.use_fixation, dtype=np.int8),
            fixationShape=np.array(self.fixation_shape),
            fixationSize=np.array(self._get_fixation_size(), dtype=np.int32),
            fixationColor=np.array(self.fixation_color, dtype=np.float32),
            fixationAlpha=np.array(self.fixation_alpha, dtype=np.float32),
            grayscaleMethod=np.array(self.grayscale_method),
            normalizeToUnit=np.array(self.normalize_to_unit, dtype=np.int8),
            pyramidInputSize=np.array(self.pyramid_input_size, dtype=np.int32),
            numOrientations=np.array(self.num_orientations, dtype=np.int32),
            bandwidth=np.array(self.bandwidth, dtype=np.float32),
            dims=np.array(self.dims, dtype=np.int32),
            normResp=np.array(norm_resp, dtype=np.int8),
        )

        if img_num % 100 == 0:
            print(f"Saved {save_path} ({time.time() - start_time:.2f}s)")

    # ------------------------------------------------------------
    # Batch processing
    # ------------------------------------------------------------

    def _load_or_init_progress(self):
        log_file = os.path.join(self.output_folder, "progress_log.json")

        if not os.path.exists(log_file):
            progress_data = {}
            with open(log_file, "w") as f:
                json.dump(progress_data, f, indent=4)
            return log_file, progress_data

        try:
            with open(log_file, "r") as f:
                progress_data = json.load(f)
        except json.JSONDecodeError:
            print("Warning: progress log corrupted. Resetting.")
            progress_data = {}
            with open(log_file, "w") as f:
                json.dump(progress_data, f, indent=4)

        return log_file, progress_data

    def process_batch(self, start_idx, end_idx, log_every=100):
        batch_start = time.time()
        log_file, progress_data = self._load_or_init_progress()

        if self.allowed_img_indices is not None:
            base_indices = self.allowed_img_indices
        else:
            base_indices = list(range(self.max_images))

        end_idx = min(end_idx, len(base_indices))
        batch_indices = base_indices[start_idx:end_idx]

        print(
            f"Processing positions {start_idx}:{end_idx} "
            f"({len(batch_indices)} images)"
        )

        with h5py.File(self.stim_filename, "r") as f:
            imgbrick = f["imgBrick"]

            for i, img_num in enumerate(batch_indices, start=1):
                save_path = os.path.join(self.output_folder, f"pyrImg{img_num}.npz")

                if str(img_num) in progress_data and os.path.exists(save_path):
                    continue

                try:
                    self.process_image(img_num, imgbrick=imgbrick)
                    progress_data[str(img_num)] = "processed"

                    if i % log_every == 0:
                        with open(log_file, "w") as fw:
                            json.dump(progress_data, fw, indent=4)
                        print(f"[batch] {i}/{len(batch_indices)} saved progress")

                except Exception as e:
                    print(f"Error processing image {img_num}: {e}")

        with open(log_file, "w") as fw:
            json.dump(progress_data, fw, indent=4)

        print(
            f"Batch completed in {time.time() - batch_start:.2f} sec."
        )

    def process_all_in_batches(self, batch_size=5000, log_every=1000):
        total_start = time.time()
        log_file, progress_data = self._load_or_init_progress()

        if self.allowed_img_indices is not None:
            indices = self.allowed_img_indices
        else:
            indices = list(range(self.max_images))

        n_imgs = len(indices)
        total_batches = (n_imgs + batch_size - 1) // batch_size

        print(
            f"Total images to process: {n_imgs}. "
            f"Batch size: {batch_size}. "
            f"Total batches: {total_batches}."
        )

        for batch_idx in range(total_batches):
            start = batch_idx * batch_size
            end = min((batch_idx + 1) * batch_size, n_imgs)

            batch_indices = indices[start:end]

            already_done = all(
                str(img) in progress_data
                and os.path.exists(os.path.join(self.output_folder, f"pyrImg{img}.npz"))
                for img in batch_indices
            )

            if already_done:
                print(f"Skipping batch {batch_idx}: already processed")
                continue

            print(f"=== Batch {batch_idx + 1}/{total_batches}: {start}:{end} ===")
            self.process_batch(start, end, log_every=log_every)

        print(f"Finished all batches in {time.time() - total_start:.2f} sec.")


# ============================================================
# Visualization helper
# ============================================================

def visualize_one_processed_npz(npz_path, title=""):
    d = np.load(npz_path, allow_pickle=True)
    bigImg = d["bigImg"]

    plt.figure(figsize=(5, 5))
    plt.imshow(bigImg, cmap="gray")
    plt.title(title or os.path.basename(npz_path))
    plt.axis("off")
    plt.show()


# ============================================================
# Example configurations
# ============================================================

CONFIG_256_NO_BACKGROUND = dict(
    interp_img_size=256,
    resize_method="resize",

    use_background=False,

    use_fixation=True,
    fixation_shape="square",
    fixation_size=None,          # auto-scales 17/714 to about 7 pixels at 256
    fixation_color=(255, 0, 0),
    fixation_alpha=0.5,

    grayscale_method="bt709",
    normalize_to_unit=False,

    pyramid_input_size=256,
    num_orientations=4,
    num_levels=4,
    bandwidth=1,
)


CONFIG_512_WITH_BACKGROUND = dict(
    interp_img_size=714,
    resize_method="remap",

    use_background=True,
    background_size=1024,
    background_color=127,

    use_fixation=True,
    fixation_shape="square",
    fixation_size=17,
    fixation_color=(255, 0, 0),
    fixation_alpha=0.5,

    grayscale_method="mean_rgb",
    normalize_to_unit=False,

    pyramid_input_size=512,
    num_orientations=8,
    num_levels="auto",
    bandwidth=1,
)


# ============================================================
# Main example
# ============================================================

if __name__ == "__main__":

    hdf5_file = "D:/NSD/stimuli/nsd_stimuli.hdf5"
    expdesign_mat = "D:/NSD/experiments/nsd_expdesign.mat"

    output_folder = "D:/NSD/pyramid_expand/configurable_256_no_background/"

    processor = StimulusPyramidProcessor(
        stim_filename=hdf5_file,
        output_folder=output_folder,
        max_images=73000,
        **CONFIG_256_NO_BACKGROUND,
    )

    # Option 1: one subject
    # processor.set_allowed_images_from_design(
    #     expdesign_mat,
    #     subject_id=0,
    # )

    # Option 2: union across all subjects
    processor.set_allowed_images_from_design(
        expdesign_mat,
        union_across_subjects=True,
    )

    processor.process_all_in_batches(
        batch_size=5000,
        log_every=1000,
    )

    print("DONE.")

## Visualize Example Pyramid Output

Load a saved `.npz` file and inspect the generated steerable pyramid features.

This section displays:

- The image that was used as input to the pyramid (`bigImg`)
- Summed orientation responses across levels (`sumOri`)
- Orientation-specific responses (`modelOri`)
- Processing parameters stored in the file

This visualization can be used to verify that preprocessing, fixation markers, background padding, grayscale conversion, and pyramid decomposition were performed as expected.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# Settings
# ============================================================

folder = r"D:/NSD/pyramid_expand/all_subjects_henderson256/"

# Choose whether to inspect a specific image or a random existing file
USE_RANDOM_FILE = False
IMAGE_ID = 22920

if USE_RANDOM_FILE:
    files = [
        f for f in os.listdir(folder)
        if f.startswith("pyrImg") and f.endswith(".npz")
    ]
    chosen_file = random.choice(files)
    p = os.path.join(folder, chosen_file)
    image_label = chosen_file.replace(".npz", "")
else:
    p = os.path.join(folder, f"pyrImg{IMAGE_ID}.npz")
    image_label = f"pyrImg{IMAGE_ID}"


# ============================================================
# Load file
# ============================================================

if not os.path.exists(p):
    raise FileNotFoundError(f"File does not exist: {p}")

with np.load(p, allow_pickle=True) as d:
    bigImg = d["bigImg"]
    sumOri = d["sumOri"]
    modelOri = d["modelOri"]

    numLevels = int(d["numLevels"])
    numOri = int(d["numOrientations"])

    # New configurable-pipeline metadata
    useBackground = bool(d["useBackground"]) if "useBackground" in d else None
    useFixation = bool(d["useFixation"]) if "useFixation" in d else None
    pyramidInputSize = int(d["pyramidInputSize"]) if "pyramidInputSize" in d else None
    normalizeToUnit = bool(d["normalizeToUnit"]) if "normalizeToUnit" in d else None

    grayscaleMethod = str(d["grayscaleMethod"]) if "grayscaleMethod" in d else "unknown"
    fixationShape = str(d["fixationShape"]) if "fixationShape" in d else "unknown"
    fixationSize = int(d["fixationSize"]) if "fixationSize" in d else None
    fixationAlpha = float(d["fixationAlpha"]) if "fixationAlpha" in d else None


# ============================================================
# Print checks
# ============================================================

print("file:", p)
print()
print("bigImg:", bigImg.shape, bigImg.dtype)
print("bigImg min/max:", float(bigImg.min()), float(bigImg.max()))
print("sumOri:", sumOri.shape, sumOri.dtype)
print("modelOri length:", len(modelOri))
print("numLevels:", numLevels)
print("numOrientations:", numOri)

print()
print("Configuration stored in file:")
print("useBackground:", useBackground)
print("useFixation:", useFixation)
print("fixationShape:", fixationShape)
print("fixationSize:", fixationSize)
print("fixationAlpha:", fixationAlpha)
print("grayscaleMethod:", grayscaleMethod)
print("normalizeToUnit:", normalizeToUnit)
print("pyramidInputSize:", pyramidInputSize)


# ============================================================
# Show input image that entered the pyramid
# ============================================================

plt.figure(figsize=(4, 4))
plt.imshow(bigImg, cmap="gray")
plt.title(f"Input to pyramid: {image_label}")
plt.axis("off")
plt.show()


# ============================================================
# Show summed orientation maps: levels + lowpass + highpass
# ============================================================

n_maps = sumOri.shape[0]

fig, axes = plt.subplots(
    1,
    n_maps,
    figsize=(4 * n_maps, 4)
)

if n_maps == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.imshow(sumOri[i], cmap="gray")

    if i < numLevels:
        title = f"sumOri\nLevel {i}"
    elif i == numLevels:
        title = "sumOri\nLowpass"
    else:
        title = "sumOri\nHighpass"

    ax.set_title(title)
    ax.axis("off")

plt.suptitle(f"Summed orientation responses: {image_label}", fontsize=14)
plt.tight_layout()
plt.show()


# ============================================================
# Show orientation-specific maps for one level
# ============================================================

level = 0  # change to 0, 1, 2, 3, ...

if level >= numLevels:
    raise ValueError(f"level must be between 0 and {numLevels - 1}")

fig, axes = plt.subplots(
    1,
    numOri,
    figsize=(4 * numOri, 4)
)

if numOri == 1:
    axes = [axes]

for o, ax in enumerate(axes):
    ax.imshow(modelOri[level][o], cmap="gray")
    ax.set_title(f"Level {level}\nOrientation {o}")
    ax.axis("off")

plt.suptitle(f"Orientation-specific responses: {image_label}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Run on a tiny subset (e.g., 20 images)
'''Process only images 0–49, in small batches of 20,
Save to a separate folder (test_run/) so it doesn’t mix with the “real” run.'''

if __name__ == "__main__":
    output_folder = "D:/NSD/pyramid_expand/test_run/"
    processor = NSDStimulusProcessor(
        hdf5_file,
        output_folder=output_folder,
        max_images=50,   # <- just first 50 images
        allow_high_levels=False
    )

    # You can skip restricting to nsd_expdesign for the test if you want:
    # processor.set_allowed_images_from_design("D:/NSD/experiments/nsd_expdesign.mat")

    processor.process_all_in_batches(batch_size=20)



# Inspect one of the output .npz files
'''You want to see:
bigImg → (512, 512)
len(sumOri) → numLevels + 2
sumOri[i] → (512, 512)
modelOri[i] → (num_orientations, 512, 512) i.e. (8, 512, 512)'''

import numpy as np
from pathlib import Path

npz_path = Path("D:/NSD/pyramid_expand/test_run/pyrImg0.npz")
data = np.load(npz_path, allow_pickle=True)

print(data.files)
print("bigImg:", data["bigImg"].shape, data["bigImg"].dtype)
print("numLevels:", data["numLevels"])
print("sumOri length:", len(data["sumOri"]))
print("modelOri length:", len(data["modelOri"]))
print("sumOri[0] shape:", data["sumOri"][0].shape)
print("modelOri[0].shape:", data["modelOri"][0].shape)


visualize_image_with_check(processor, img_num=0)

#if good switch to:

output_folder = "D:/NSD/pyramid_expand/all_subjects/"
processor = NSDStimulusProcessor(
    hdf5_file,
    output_folder=output_folder,
    max_images=73000
)
processor.set_allowed_images_from_design("D:/NSD/experiments/nsd_expdesign.mat")
processor.process_all_in_batches(batch_size=30000)
